# FTL Syria — Linear Regression, Cost Function & Gradient Descent
**Live demo notebook | Google Colab**

**Question:** Can we predict a house price from its size, and how does a model learn a good-fitting line?

This demo follows the slides: the linear model `f(x) = w*x + b`, prediction errors, the cost function, gradients, simultaneous parameter updates, learning rate, and convergence.

**Dataset:** The slides show house size and price. We use a small illustrative dataset in the same style (size in square feet; price in thousands of USD). It is synthetic teaching data, not real estate data.

## 1. Load libraries and inspect the data

Each row is one training example. `x` is the input feature (house size), and `y` is the target (price).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Make the example reproducible.
np.random.seed(7)

# Illustrative house sizes in square feet.
x = np.array([850, 1000, 1200, 1400, 1550, 1700, 1850, 2000, 2200, 2400], dtype=float)

# Create prices with an upward trend plus a little random variation.
# Price is measured in thousands of US dollars.
y = 0.16 * x + 60 + np.random.normal(0, 18, size=len(x))

# A table makes the input feature and target easy to inspect.
data = pd.DataFrame({"size_sqft": x, "price_thousands_usd": y})
display(data.round(1))
print("Number of examples m =", len(x))

In [ ]:
# Plot the examples: each dot is one observed house.
plt.figure(figsize=(8, 5))
plt.scatter(x, y, s=70)
plt.xlabel("House size (square feet)")
plt.ylabel("Price (thousands of USD)")
plt.title("House size and price")
plt.grid(alpha=0.3)
plt.show()

## 2. The linear regression model

`f(x) = w*x + b`

- `w`: weight or slope — how much the prediction changes as house size changes.
- `b`: bias or intercept — the predicted value when `x = 0`.
- `y_hat`: the model's prediction.

In [ ]:
def predict(x, w, b):
    # Apply the line to one input or an array of inputs.
    return w * x + b

# Start with a guess for the slope and intercept.
w, b = 0.10, 100
y_hat = predict(x, w, b)

comparison = pd.DataFrame({
    "size_sqft": x,
    "actual_price": y,
    "predicted_price": y_hat,
    "prediction_error": y_hat - y
})
display(comparison.round(1))

# Use many x values to draw a smooth line.
x_line = np.linspace(x.min(), x.max(), 100)
plt.figure(figsize=(8, 5))
plt.scatter(x, y, s=65, label="Observed houses")
plt.plot(x_line, predict(x_line, w, b), linewidth=2, label=f"Initial line: w={w}, b={b}")
plt.xlabel("House size (square feet)")
plt.ylabel("Price (thousands of USD)")
plt.title("An initial guess")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

## 3. Cost function: Mean Squared Error

The error for example `i` is `prediction - actual`. Squaring prevents positive and negative errors cancelling out, and gives larger errors more influence.

The slides use this convention:

\[
J(w,b)=\frac{1}{2m}\sum_{i=1}^{m}(wx^{(i)}+b-y^{(i)})^2
\]

The factor `1/2` makes the derivatives simpler; it does not change which parameters minimise the cost.

In [ ]:
def compute_cost(x, y, w, b):
    # m is the number of training examples.
    m = len(x)
    predictions = predict(x, w, b)   # Calculate predictions for every example.
    errors = predictions - y          # Compare predictions with actual targets.
    squared_errors = errors ** 2      # Square each error.
    return np.sum(squared_errors) / (2 * m)  # Average with the slide's 1/(2m).

print(f"Cost for the initial line: {compute_cost(x, y, w, b):.2f}")

### Compare different candidate lines

Each line has its own `w` and `b`. The cost function gives one number to summarise how well each line fits all the examples.

In [ ]:
# Try several slopes and intercepts, like the different lines in the slides.
candidates = [(0.00, 250), (0.10, 100), (0.16, 60), (0.22, 20)]
rows = []

plt.figure(figsize=(9, 5))
plt.scatter(x, y, color="black", s=60, label="Observed houses")
for w_try, b_try in candidates:
    cost = compute_cost(x, y, w_try, b_try)
    rows.append({"w": w_try, "b": b_try, "cost J(w,b)": cost})
    plt.plot(x_line, predict(x_line, w_try, b_try),
             label=f"w={w_try:.2f}, b={b_try}, cost={cost:.0f}")

plt.xlabel("House size (square feet)")
plt.ylabel("Price (thousands of USD)")
plt.title("Different lines, different costs")
plt.grid(alpha=0.3)
plt.legend()
plt.show()
display(pd.DataFrame(rows).sort_values("cost J(w,b)").round(2))

## 4. Gradient descent: how parameters learn

For the cost function above:

\[
\frac{\partial J}{\partial w}=\frac{1}{m}\sum_{i=1}^{m}(wx^{(i)}+b-y^{(i)})x^{(i)}
\]

\[
\frac{\partial J}{\partial b}=\frac{1}{m}\sum_{i=1}^{m}(wx^{(i)}+b-y^{(i)})
\]

Update the parameters by moving opposite the gradient:

\[
w := w-\alpha\frac{\partial J}{\partial w},\qquad
b := b-\alpha\frac{\partial J}{\partial b}
\]

`alpha` is the learning rate. **Simultaneous update:** calculate both gradients from the same old values of `w` and `b`, then update both parameters.

In [ ]:
def compute_gradients(x, y, w, b):
    # Batch gradients use all examples for each update.
    m = len(x)
    predictions = predict(x, w, b)
    errors = predictions - y

    # The slope gradient includes x: changing w affects predictions in proportion to x.
    dj_dw = np.sum(errors * x) / m

    # The intercept gradient is the average prediction error.
    dj_db = np.sum(errors) / m
    return dj_dw, dj_db

### One update, step by step

House sizes are large numbers, so using raw square feet can make the slope gradient very large. We rescale the feature to **thousands of square feet** to make the gradient descent demo more numerically stable. This changes the units of the slope, not the underlying relationship.

In [ ]:
x_scaled = x / 1000.0  # For example, 2000 square feet becomes 2.0.

w_old, b_old = 0.0, 0.0
alpha = 0.1

# Both gradients are computed from the same old parameter values.
dj_dw, dj_db = compute_gradients(x_scaled, y, w_old, b_old)

# Compute both new values before replacing either old value.
w_new = w_old - alpha * dj_dw
b_new = b_old - alpha * dj_db

print(f"Old parameters: w={w_old:.4f}, b={b_old:.4f}")
print(f"Gradients: dJ/dw={dj_dw:.4f}, dJ/db={dj_db:.4f}")
print(f"New parameters: w={w_new:.4f}, b={b_new:.4f}")
print(f"Cost before update: {compute_cost(x_scaled, y, w_old, b_old):.2f}")
print(f"Cost after update:  {compute_cost(x_scaled, y, w_new, b_new):.2f}")

## 5. Implement batch gradient descent

Batch gradient descent uses the **whole dataset** to calculate each update. One update is one iteration.

In [ ]:
def gradient_descent(x, y, w_init=0.0, b_init=0.0, alpha=0.1, iterations=2000):
    # Start from the supplied initial parameter values.
    w, b = float(w_init), float(b_init)
    cost_history, w_history, b_history = [], [], []

    for _ in range(iterations):
        # Step 1: compute both gradients using current w and b.
        dj_dw, dj_db = compute_gradients(x, y, w, b)

        # Step 2: calculate both updates from the same old values (simultaneously).
        w_next = w - alpha * dj_dw
        b_next = b - alpha * dj_db

        # Step 3: replace both parameters only after both new values exist.
        w, b = w_next, b_next

        # Save values so we can inspect how training progresses.
        cost_history.append(compute_cost(x, y, w, b))
        w_history.append(w)
        b_history.append(b)

    return w, b, cost_history, w_history, b_history

w_fit, b_fit, costs, w_hist, b_hist = gradient_descent(
    x_scaled, y, alpha=0.1, iterations=2000
)

print(f"Learned w: {w_fit:.3f} thousand USD per 1,000 sqft")
print(f"Learned b: {b_fit:.3f} thousand USD")
print(f"Cost at first recorded update: {costs[0]:.2f}")
print(f"Final cost: {costs[-1]:.2f}")

In [ ]:
# A falling curve indicates that the cost is getting smaller during training.
plt.figure(figsize=(8, 4))
plt.plot(costs)
plt.xlabel("Iteration")
plt.ylabel("Cost J(w,b)")
plt.title("Cost during gradient descent")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# Compare the learned line with the observations.
plt.figure(figsize=(8, 5))
plt.scatter(x_scaled, y, s=65, label="Observed houses")
plt.plot(x_scaled, predict(x_scaled, w_fit, b_fit), linewidth=2.5, label="Learned line")
plt.xlabel("House size (1,000 square feet)")
plt.ylabel("Price (thousands of USD)")
plt.title("The model after gradient descent")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

## 6. Learning rate: too small vs. reasonable vs. too large

The learning rate `alpha` controls step size:
- Too small: cost may decrease very slowly.
- Reasonable: cost decreases steadily.
- Too large: cost may oscillate or diverge.

Try adjusting the learning rates below.

In [ ]:
learning_rates = [0.001, 0.1, 0.8]
plt.figure(figsize=(9, 5))

for alpha_try in learning_rates:
    _, _, cost_try, _, _ = gradient_descent(
        x_scaled, y, alpha=alpha_try, iterations=300
    )
    # A log scale helps compare curves with different cost magnitudes.
    plt.plot(np.maximum(cost_try, 1e-10), label=f"alpha={alpha_try}")

plt.yscale("log")
plt.xlabel("Iteration")
plt.ylabel("Cost (log scale)")
plt.title("How the learning rate changes training")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

## 7. Visualise the cost surface from above

The contour levels show the cost for different parameter pairs `(w, b)`. The gradient descent path moves toward lower-cost regions.

In [ ]:
# Create candidate values for the slope and intercept.
w_values = np.linspace(-100, 300, 100)
b_values = np.linspace(-100, 300, 100)
W, B = np.meshgrid(w_values, b_values)
J = np.zeros_like(W)

# Evaluate cost for each parameter pair in the grid.
for i in range(W.shape[0]):
    for j in range(W.shape[1]):
        J[i, j] = compute_cost(x_scaled, y, W[i, j], B[i, j])

plt.figure(figsize=(8, 6))
contours = plt.contour(W, B, J, levels=25)
plt.clabel(contours, inline=True, fontsize=8)
plt.plot(w_hist, b_hist, marker=".", markersize=2, linewidth=1, label="Gradient descent path")
plt.scatter([w_fit], [b_fit], marker="*", s=180, label="Final parameters")
plt.xlabel("w (slope)")
plt.ylabel("b (intercept)")
plt.title("Cost contours and learning path")
plt.legend()
plt.grid(alpha=0.2)
plt.show()

## 8. Predict for a new house

The model used size in thousands of square feet and predicts price in thousands of USD.

In [ ]:
new_size_sqft = 2000
new_size_scaled = new_size_sqft / 1000.0
predicted_price_thousands = predict(new_size_scaled, w_fit, b_fit)

print(f"House size: {new_size_sqft:,} square feet")
print(f"Predicted price: ${predicted_price_thousands:.1f} thousand")
print(f"Predicted price: ${predicted_price_thousands * 1000:,.0f}")

## 9. Student activity

Before running the cell, ask learners to predict what will happen.

1. If the slope `w` is too small, how will the line look?
2. If `b` increases while `w` stays fixed, what changes?
3. Why square the errors in MSE?
4. What can happen if the learning rate is too large?
5. Why calculate both gradients before updating either parameter?

Change `student_w` and `student_b` and see how the cost changes.

In [ ]:
student_w = 100.0  # Change this value to try a different slope.
student_b = 100.0  # Change this value to try a different intercept.
student_cost = compute_cost(x_scaled, y, student_w, student_b)
print(f"Cost for your line: {student_cost:.2f}")

plt.figure(figsize=(8, 5))
plt.scatter(x_scaled, y, s=65, label="Observed houses")
plt.plot(x_scaled, predict(x_scaled, student_w, student_b), linewidth=2,
         label=f"Your line: w={student_w:.1f}, b={student_b:.1f}")
plt.xlabel("House size (1,000 square feet)")
plt.ylabel("Price (thousands of USD)")
plt.title("Student experiment")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

## Recap

- **Model:** `f(x) = w*x + b`.
- **Cost function:** summarises errors across the examples.
- **MSE:** squares errors and averages them; these slides use the `1/(2m)` convention.
- **Gradient:** measures how the cost changes as a parameter changes.
- **Gradient descent:** repeatedly moves parameters opposite the gradient.
- **Learning rate:** controls the step size.
- **Simultaneous update:** calculate both gradients using the old parameters before updating `w` and `b`.

**Closing question:** If two lines both look reasonable, how does the cost function help compare them objectively?